# Module 22: ML Project Architecture
## Lesson: Building Production-Ready ML Systems

This lesson covers the architectural patterns, tools, and practices for
designing ML projects that are scalable, maintainable, and production-ready.
We move from notebook prototypes to modular, configurable, and tracked systems.

In [ ]:
import yaml
import os
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

print("All imports successful")

### 1. Project Structure Patterns

Two dominant patterns exist for ML projects:

**Cookiecutter Data Science**: Opinionated, standardized layout with clear
separation between notebooks, source code, data, and models. Best for
data-heavy exploratory projects that need to become production systems.

**Modular Monolith**: Domain-organized packages (ingestion, features,
training, serving). Better for teams where different members own different
parts of the ML lifecycle.

In [ ]:
def print_ascii_tree(structure, indent=0):
    """Print a directory structure as an ASCII tree."""
    for key, value in structure.items():
        prefix = "|   " * indent + "|-- " if indent > 0 else ""
        print(f"{prefix}{key}/" if isinstance(value, dict) else f"{prefix}{key}")
        if isinstance(value, dict):
            print_ascii_tree(value, indent + 1)


cookiecutter_ds = {
    "data": {"raw": None, "processed": None, "interim": None, "external": None},
    "notebooks": None,
    "src": {
        "data": ["loader.py", "validator.py"],
        "features": ["engineering.py", "selectors.py"],
        "models": ["trainer.py", "evaluator.py", "predictor.py"],
        "visualization": ["plots.py"],
    },
    "models": None,
    "config": ["default.yaml"],
    "tests": ["test_data.py", "test_features.py", "test_model.py"],
}

print("Cookiecutter Data Science Structure:")
print_ascii_tree(cookiecutter_ds)

### 2. Configuration Management with YAML and pydantic

Separating configuration from code is critical for reproducibility.
We use YAML for defaults and pydantic-settings for environment-aware config.

In [ ]:
# Define configuration schema
@dataclass
class DataConfig:
    raw_path: str = "data/raw/dataset.csv"
    test_size: float = 0.2
    random_state: int = 42


@dataclass
class ModelConfig:
    name: str = "random_forest"
    params: Dict[str, Any] = field(default_factory=lambda: {
        "n_estimators": 100,
        "max_depth": 10,
    })


@dataclass
class MLConfig:
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    experiment_name: str = "churn_prediction"
    tracking_uri: str = "http://localhost:5000"


def load_config(path="config.yaml") -> MLConfig:
    """Load config from YAML file with defaults."""
    config = MLConfig()
    if os.path.exists(path):
        with open(path) as f:
            data = yaml.safe_load(f)
        if data:
            if "data" in data:
                config.data = DataConfig(**data["data"])
            if "model" in data:
                config.model = ModelConfig(**data["model"])
            if "experiment_name" in data:
                config.experiment_name = data["experiment_name"]
    return config


# Demonstrate
sample_config = {
    "data": {"raw_path": "s3://bucket/churn.csv", "test_size": 0.3},
    "model": {"name": "xgboost", "params": {"n_estimators": 200}},
    "experiment_name": "churn_v2",
}
with open("/tmp/config.yaml", "w") as f:
    yaml.dump(sample_config, f)

cfg = load_config("/tmp/config.yaml")
print(f"Data path: {cfg.data.raw_path}")
print(f"Model: {cfg.model.name}")
print(f"Experiment: {cfg.experiment_name}")

### 3. Experiment Tracking with MLflow

Experiment tracking systems log parameters, metrics, and artifacts.
MLflow is the most widely adopted open-source option.

In [ ]:
# Simulated MLflow tracking (no server needed)
import hashlib
import datetime


class LocalExperimentTracker:
    """Simple experiment tracker simulating MLflow API."""

    def __init__(self, experiment_name="default"):
        self.experiment_name = experiment_name
        self.run_id = hashlib.md5(str(datetime.datetime.now()).encode()).hexdigest()[:8]
        self.params = {}
        self.metrics = {}
        self.artifacts = []

    def log_param(self, key, value):
        self.params[key] = value
        print(f"[TRACKER] param: {key} = {value}")

    def log_metric(self, key, value):
        self.metrics[key] = value
        print(f"[TRACKER] metric: {key} = {value:.4f}")

    def log_artifact(self, path):
        self.artifacts.append(path)
        print(f"[TRACKER] artifact: {path}")

    def get_summary(self):
        return {
            "experiment": self.experiment_name,
            "run_id": self.run_id,
            "params": self.params,
            "metrics": self.metrics,
        }


tracker = LocalExperimentTracker("housing_prices")
tracker.log_param("n_estimators", 100)
tracker.log_param("max_depth", 10)
tracker.log_metric("rmse", 45000.0)
tracker.log_metric("r2", 0.87)
tracker.log_artifact("feature_importance.png")
print()
print("Run summary:")
for k, v in tracker.get_summary().items():
    print(f"  {k}: {v}")

### 4. Data Versioning with DVC

Data Version Control (DVC) extends Git for data and ML models.
It creates lightweight pointer files (`.dvc`) that track data versions in Git,
while the actual data lives in remote storage (S3, GCS, etc.).

In [ ]:
# DVC usage simulation
print("DVC Workflow:")
print("  $ dvc init")
print("  $ dvc add data/raw/churn.csv")
print("  $ git add data/raw/churn.csv.dvc .gitignore")
print("  $ git commit -m 'Add churn dataset v1'")
print("  $ dvc remote add -d myremote s3://ml-data-bucket/churn")
print("  $ dvc push")
print()
print("To switch to a different data version:")
print("  $ git checkout <commit-hash>")  # switches .dvc file
print("  $ dvc checkout")  # syncs actual data
print()
print("DVC metafile (churn.csv.dvc) contents:")
dvc_meta = {
    "md5": "a1b2c3d4e5f6...",
    "size": 452890,
    "path": "data/raw/churn.csv",
    "remote": "s3://ml-data-bucket/churn",
}
print(json.dumps(dvc_meta, indent=2))

### 5. Model Registry

A model registry manages model lifecycle: registration, versioning,
staging, and promotion to production. MLflow has a built-in registry.

In [ ]:
@dataclass
class ModelRegistryEntry:
    name: str
    version: int
    stage: str = "Staging"  # None, Staging, Production, Archived
    run_id: Optional[str] = None
    metrics: Dict[str, float] = field(default_factory=dict)
    path: Optional[str] = None


class ModelRegistry:
    """Simple model registry for tracking model versions."""

    def __init__(self):
        self._models: Dict[str, List[ModelRegistryEntry]] = {}

    def register(self, name: str, path: str, metrics: Dict) -> ModelRegistryEntry:
        if name not in self._models:
            self._models[name] = []
        version = len(self._models[name]) + 1
        entry = ModelRegistryEntry(
            name=name, version=version, path=path, metrics=metrics
        )
        self._models[name].append(entry)
        print(f"Registered {name} v{version} (stage: {entry.stage})")
        return entry

    def promote_to_production(self, name: str, version: int):
        for entry in self._models.get(name, []):
            entry.stage = "Archived" if entry.stage == "Production" else entry.stage
        for entry in self._models.get(name, []):
            if entry.version == version:
                entry.stage = "Production"
                print(f"Promoted {name} v{version} to Production")
                return entry
        return None

    def get_production(self, name: str) -> Optional[ModelRegistryEntry]:
        for entry in self._models.get(name, []):
            if entry.stage == "Production":
                return entry
        return None


registry = ModelRegistry()
m1 = registry.register("churn_model", "models/v1/model.pkl", {"f1": 0.85})
m2 = registry.register("churn_model", "models/v2/model.pkl", {"f1": 0.88})
registry.promote_to_production("churn_model", 2)
prod = registry.get_production("churn_model")
print(f"Production model: {prod.name} v{prod.version}, metrics={prod.metrics}")

### 6. Pipeline Orchestration

Orchestration tools schedule and manage ML pipelines. Airflow (DAG-based)
and Prefect (Pythonic) are the most popular choices. Key concepts:
- Tasks: individual units of work
- DAG: directed acyclic graph defining task dependencies
- Scheduler: triggers runs on schedule or events
- Executor: runs tasks (locally, distributed, or in cloud)

In [ ]:
# Airflow-style DAG definition (for illustration)
pipeline_steps = {
    "ingest": {
        "depends_on": [],
        "function": "load_data_from_source",
        "retries": 3,
    },
    "validate": {
        "depends_on": ["ingest"],
        "function": "validate_data_schema",
        "retries": 1,
    },
    "engineer_features": {
        "depends_on": ["validate"],
        "function": "create_features",
        "retries": 2,
    },
    "train": {
        "depends_on": ["engineer_features"],
        "function": "train_model",
        "retries": 1,
    },
    "evaluate": {
        "depends_on": ["train"],
        "function": "evaluate_and_promote",
        "retries": 0,
    },
    "deploy": {
        "depends_on": ["evaluate"],
        "function": "deploy_to_production",
        "retries": 0,
    },
}

print("Pipeline DAG:")
for step, config in pipeline_steps.items():
    deps = config["depends_on"] if config["depends_on"] else ["START"]
    print(f"  {step} <- {', '.join(deps)}")
    print(f"    runs: {config['function']}(), retries={config['retries']}")
print()
print("DAG visualization:")
print("  START -> ingest -> validate -> engineer_features")
print("                                    |")
print("                                    v")
print("                              train -> evaluate -> deploy")

### 7. Feature Stores Concept

Feature stores centralize feature computation and serving:
- **Offline store**: batch-computed features for training
- **Online store**: low-latency feature serving for inference
Popular options: Feast, Tecton, SageMaker Feature Store

In [ ]:
# Feature store concept
features_definition = {
    "user_features": {
        "tenure": {
            "type": "numerical",
            "source": "subscriptions.tenure",
            "compute": "SELECT user_id, DATEDIFF('day', start_date, CURRENT_DATE) as tenure FROM subscriptions",
        },
        "avg_monthly_charges": {
            "type": "numerical",
            "source": "billing.charges",
            "compute": "SELECT user_id, AVG(amount) as avg_monthly_charges FROM billing GROUP BY user_id",
        },
    },
}

print("Feature Store Architecture:")
print()
print("  DATA SOURCES  -->  FEATURE PIPELINE  -->  OFFLINE STORE (for training)")
print("  (raw tables)        (compute features)     (Parquet/Delta Lake)")
print("                                           -->  ONLINE STORE (for inference)")
print("                                                (Redis/DynamoDB)")
print()
print("Example feature:")
print("  name:            avg_monthly_charges")
print("  type:            numerical")
print("  source:          billing.charges")
print("  compute:         AVG(amount) GROUP BY user_id")

### 8. MLOps Maturity Model

The MLOps maturity model helps organizations assess their ML capabilities:

```
Level 0: No MLOps
  - Jupyter notebooks, manual deployment, no tracking

Level 1: DevOps for Code
  - Git, CI/CD for code, but data/models untracked

Level 2: Automated Training
  - Experiment tracking, data versioning, model registry

Level 3: Automated Deployment
  - A/B testing, automated retraining, model monitoring

Level 4: Full MLOps
  - AutoML, self-healing, automated drift detection
```

In [ ]:
def assess_maturity_level(
    has_version_control: bool,
    has_ci_cd: bool,
    has_experiment_tracking: bool,
    has_data_versioning: bool,
    has_model_registry: bool,
    has_automated_retraining: bool,
    has_drift_monitoring: bool,
) -> int:
    """Assess MLOps maturity level based on capabilities."""
    if not has_version_control:
        return 0
    if not has_experiment_tracking:
        return 1
    if has_experiment_tracking and has_data_versioning:
        if has_model_registry and has_ci_cd:
            if has_automated_retraining:
                if has_drift_monitoring:
                    return 4
                return 3
            return 2
    return 1


# Example assessment
my_team = {
    "has_version_control": True,
    "has_ci_cd": True,
    "has_experiment_tracking": True,
    "has_data_versioning": False,
    "has_model_registry": True,
    "has_automated_retraining": False,
    "has_drift_monitoring": False,
}

level = assess_maturity_level(**my_team)
print(f"Your team is at MLOps maturity Level {level}: ", end="")
descriptions = ["No MLOps", "DevOps for Code", "Automated Training",
                "Automated Deployment", "Full MLOps"]
print(descriptions[level])
print()
print("Next step: Add data versioning with DVC to reach Level 3")

### Summary

In this lesson you learned:
- Project structure patterns (cookiecutter DS, modular monolith)
- Configuration management with YAML, Hydra, and pydantic-settings
- Experiment tracking with MLflow and Weights & Biases
- Data versioning with DVC
- Model registries for lifecycle management
- Pipeline orchestration with Airflow/Prefect
- Feature stores and the online/offline paradigm
- MLOps maturity model for assessing team capabilities

**Next**: Apply these concepts in the exercises to design your own ML project architecture.